In [ ]:
%pip install -q biopython
import os, re, subprocess, tempfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict

base_dir = Path("/home/bulat_kharisov/inf")
(base_dir / "results" / "bed").mkdir(parents=True, exist_ok=True)
(base_dir / "scripts").mkdir(parents=True, exist_ok=True)
os.chdir(base_dir)

from Bio import SeqIO
print("base_dir:", base_dir, "| cwd:", os.getcwd())

In [ ]:
FASTA_IN  = base_dir / "data" / "genome.fa"
G4_BED    = base_dir / "results" / "bed" / "g4.bed"
ZHUNT_BED = base_dir / "results" / "bed" / "zhunt.bed"

CHROMS  = None
MAX_BP  = None


G4_PLUS  = re.compile(r"(?:G{3,5}[ACGT]{1,7}){3,}G{3,5}")
G4_MINUS = re.compile(r"(?:C{3,5}[ACGT]{1,7}){3,}C{3,5}")


ZHUNT_WIN, ZHUNT_MIN, ZHUNT_MAX = 12, 8, 12
Z_THRESHOLD   = 400.0
ZHUNT_CHUNK   = 2_000_000
ZHUNT_OVERLAP = 1_000
ZHUNT_PROC    = 96
ZHUNT_SRC_URL = "https://raw.githubusercontent.com/diegopenilla/FlaskHunt/master/zhunt3.c"

assert FASTA_IN.exists(), f"нет генома {FASTA_IN}"

In [ ]:
seqs = {}
for rec in SeqIO.parse(str(FASTA_IN), "fasta"):
    if CHROMS is not None and rec.id not in CHROMS:
        continue
    s = str(rec.seq).upper()
    if MAX_BP is not None:
        s = s[:MAX_BP]
    seqs[rec.id] = s
    if CHROMS is not None and len(seqs) == len(CHROMS):
        break
print("загружено:", {k: len(v) for k, v in seqs.items()})

In [ ]:
def find_g4(seq, chrom):
    rows = []
    for strand, pat in (("+", G4_PLUS), ("-", G4_MINUS)):
        for m in pat.finditer(seq):
            rows.append((chrom, m.start(), m.end(), strand))
    return rows

n = 0
with open(G4_BED, "w") as bed:
    for chrom, s in seqs.items():
        rows = find_g4(s, chrom)
        rows.sort(key=lambda r: (r[1], r[2]))
        for c, st, en, strand in rows:
            bed.write(f"{c}\t{st}\t{en}\tG4\t{en - st}\t{strand}\n")
            n += 1
print("готово ->", G4_BED, "| G4-интервалов:", n)

In [ ]:
zhunt_src = base_dir / "scripts" / "zhunt3.c"
zhunt_bin = base_dir / "scripts" / "zhunt3"
if not zhunt_src.exists():
    subprocess.run(["wget", "-q", "-O", str(zhunt_src), ZHUNT_SRC_URL], check=True)
subprocess.run(["gcc", str(zhunt_src), "-o", str(zhunt_bin), "-lm"], check=True)
print("zhunt собран:", zhunt_bin)

In [ ]:
def zhunt_chunk(task):
    chrom, gstart, owned_end, sub = task
    acgt = sum(sub.count(b) for b in "ACGT")
    if acgt == 0:
        return chrom, []
    if acgt == len(sub):
        seq_clean = sub
        gmap = lambda i: gstart + i
    else:
        keep = [j for j, b in enumerate(sub) if b in "ACGT"]
        seq_clean = "".join(sub[j] for j in keep)
        gmap = lambda i: gstart + keep[i]
    with tempfile.NamedTemporaryFile("w", suffix=".seq",
                                     dir=str(base_dir / "scripts"), delete=False) as tf:
        tf.write(seq_clean)
        tmp = tf.name
    try:
        subprocess.run([str(zhunt_bin), str(ZHUNT_WIN), str(ZHUNT_MIN), str(ZHUNT_MAX), tmp],
                       check=True, stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
        out = subprocess.run(["awk", "-v", f"t={Z_THRESHOLD}",
                              "NR>1 && $3>t {print NR-2}", tmp + ".Z-SCORE"],
                             check=True, capture_output=True, text=True).stdout
    finally:
        for f in (tmp, tmp + ".Z-SCORE"):
            if os.path.exists(f):
                os.remove(f)
    flagged = []
    for x in out.split():
        p = gmap(int(x))
        if p < owned_end:
            flagged.append(p)
    return chrom, flagged

def merge_positions(positions):
    positions = sorted(positions)
    out = []
    if positions:
        st = pr = positions[0]
        for v in positions[1:]:
            if v <= pr + 1:
                pr = v
            else:
                out.append((st, pr + 1)); st = pr = v
        out.append((st, pr + 1))
    return out

tasks = []
for chrom, s in seqs.items():
    L = len(s)
    for st in range(0, L, ZHUNT_CHUNK):
        owned_end = min(st + ZHUNT_CHUNK, L)
        en = min(st + ZHUNT_CHUNK + ZHUNT_OVERLAP, L)
        tasks.append((chrom, st, owned_end, s[st:en]))
print(f"zhunt: кусков {len(tasks)} x ~{ZHUNT_CHUNK // 10**6} Мб | процессов {ZHUNT_PROC}")

by_chrom = defaultdict(list)
done = 0
with ThreadPoolExecutor(max_workers=ZHUNT_PROC) as ex:
    for chrom, flagged in ex.map(zhunt_chunk, tasks):
        by_chrom[chrom].extend(flagged)
        done += 1
        print(f"  {done}/{len(tasks)} кусков", end="\r")

n = 0
with open(ZHUNT_BED, "w") as bed:
    for chrom in sorted(by_chrom):
        for st, en in merge_positions(by_chrom[chrom]):
            bed.write(f"{chrom}\t{st}\t{en}\tZHUNT\t{en - st}\t.\n")
            n += 1
print("\nготово ->", ZHUNT_BED, "| zhunt-интервалов:", n)